In [20]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

X, y = make_blobs(n_samples=1000, centers=2, n_features=2, cluster_std=17.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.9, random_state=42,shuffle=True)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
y_test = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

In [21]:
class UnderfitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 2),
            nn.ReLU(),
            nn.Linear(2, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)

class OverfitMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(2, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )
    def forward(self, x):
        return self.model(x)


def train(model, optimizer, X_train, y_train, epochs, l2=0.0):
    criterion = nn.BCELoss()
    for epoch in range(epochs):
        model.train()
        y_pred = model(X_train)
        loss = criterion(y_pred, y_train)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate(model, X, y, label):
    model.eval()
    with torch.no_grad():
        y_pred = model(X)
        acc = ((y_pred > 0.5) == y).float().mean().item()
        print(f"{label} Accuracy: {acc:.4f}")


In [22]:
underfit_model = UnderfitMLP()
overfit_model = OverfitMLP()

underfit_opt = optim.SGD(underfit_model.parameters(), lr=0.1, weight_decay=10.0)
overfit_opt = optim.Adam(overfit_model.parameters(), lr=0.001, weight_decay=0.0)

train(underfit_model, underfit_opt, X_train, y_train, epochs=10)

train(overfit_model, overfit_opt, X_train, y_train, epochs=1000)

evaluate(underfit_model, X_train, y_train, "Underfit - Treino")
evaluate(underfit_model, X_test, y_test, "Underfit - Teste")


evaluate(overfit_model, X_train, y_train, "Overfit - Treino")
evaluate(overfit_model, X_test, y_test, "Overfit - Teste")


Underfit - Treino Accuracy: 0.5200
Underfit - Teste Accuracy: 0.4978
Overfit - Treino Accuracy: 1.0000
Overfit - Teste Accuracy: 0.5378
